# 🎬 Netflix Shows & Movies Data Analysis
## A Beginner Data Science Project

Welcome! In this project, we will learn how to:
1. **Load and explore** a dataset
2. **Clean** the data (handle missing values, fix data types)
3. **Analyze** Netflix shows and movies (ratings, views, genres, etc.)
4. **Visualize** the results with charts and graphs
5. **Draw conclusions** from the data

> 📌 **Tip:** Run each cell one by one by clicking on it and pressing **Shift + Enter**

---

## Step 1: Install and Import Required Libraries

In data science, we use **libraries** (pre-written code) to make our work easier.

- **pandas**: Used to work with tables of data (like Excel)
- **numpy**: Used for math operations
- **matplotlib**: Used to create charts and graphs
- **seaborn**: Makes prettier charts (built on top of matplotlib)

In [ ]:
# Install libraries if not already installed
# (Uncomment the next line if you need to install them)
# !pip install pandas numpy matplotlib seaborn

# Import the libraries we need
import pandas as pd          # for data manipulation
import numpy as np           # for numerical operations
import matplotlib.pyplot as plt  # for creating charts
import seaborn as sns        # for prettier charts

# This line makes our charts appear inside the notebook
%matplotlib inline

# Set a nice style for all our charts
available = plt.style.available
style = 'seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in available else 'ggplot'
plt.style.use(style)
sns.set_palette('Set2')

print('✅ All libraries imported successfully!')

---
## Step 2: Load the Dataset

We'll load our Netflix dataset from a CSV (Comma-Separated Values) file.
A CSV file is like a simple spreadsheet stored as text.

In [ ]:
# Load the CSV file into a DataFrame
# A DataFrame is like a table with rows and columns
df = pd.read_csv('netflix_data.csv')

print(f'✅ Dataset loaded successfully!')
print(f'📊 Total rows: {len(df)}')
print(f'📋 Total columns: {len(df.columns)}')

---
## Step 3: Explore the Dataset

Before doing any analysis, we always explore the data first to understand what we have.

In [ ]:
# Look at the first 5 rows of our dataset
# .head() shows the top rows
print('📌 First 5 rows of the dataset:')
df.head()

In [ ]:
# Look at the last 5 rows
print('📌 Last 5 rows of the dataset:')
df.tail()

In [ ]:
# Get a summary of the dataset
# .info() tells us column names, data types, and missing values
print('📌 Dataset Information:')
df.info()

In [ ]:
# Get basic statistics for numerical columns
# .describe() gives min, max, average, etc.
print('📌 Statistical Summary:')
df.describe().round(2)

In [ ]:
# Check column names
print('📌 Column Names:')
for col in df.columns:
    print(f'  - {col}')

---
## Step 4: Data Cleaning

Real data often has problems like missing values or wrong data types.
We need to clean the data before analysis.

In [ ]:
# Check for missing values in each column
print('📌 Missing Values per Column:')
missing = df.isnull().sum()
print(missing)
print(f'\nTotal missing values: {missing.sum()}')

In [ ]:
# Check for duplicate rows
duplicates = df.duplicated().sum()
print(f'📌 Number of duplicate rows: {duplicates}')

if duplicates > 0:
    df = df.drop_duplicates()
    print('✅ Duplicates removed!')
else:
    print('✅ No duplicates found!')

In [ ]:
# Separate TV Shows and Movies for easier analysis
tv_shows = df[df['type'] == 'TV Show'].copy()
movies = df[df['type'] == 'Movie'].copy()

print(f'📺 Number of TV Shows: {len(tv_shows)}')
print(f'🎬 Number of Movies: {len(movies)}')

---
## Step 5: Basic Analysis - Content Distribution

Let's start with simple questions:
- How many TV shows vs movies are there?
- What genres are most popular?

In [ ]:
# Count TV Shows vs Movies
content_counts = df['type'].value_counts()
print('📌 Content Type Distribution:')
print(content_counts)

# Create a Pie Chart
fig, ax = plt.subplots(figsize=(7, 7))
colors = ['#E50914', '#221F1F']  # Netflix red and dark color
wedge_props = {'edgecolor': 'white', 'linewidth': 2}
ax.pie(
    content_counts.values,
    labels=content_counts.index,
    autopct='%1.1f%%',
    colors=colors,
    wedgeprops=wedge_props,
    startangle=90
)
ax.set_title('Netflix Content: TV Shows vs Movies', fontsize=16, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('chart_content_type.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved as chart_content_type.png')

In [ ]:
# What are the most popular genres?
# The genre column has values like 'Sci-Fi/Horror' - let's split them
all_genres = []
for genre_str in df['genre']:
    genres = genre_str.split('/')
    all_genres.extend([g.strip() for g in genres])

# Count how many times each genre appears
from collections import Counter
genre_counts = Counter(all_genres)
genre_df = pd.DataFrame(genre_counts.items(), columns=['Genre', 'Count'])
genre_df = genre_df.sort_values('Count', ascending=False).head(12)

print('📌 Top 12 Most Popular Genres:')
print(genre_df.to_string(index=False))

# Create a Bar Chart
fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(genre_df['Genre'], genre_df['Count'], color=sns.color_palette('Set2', len(genre_df)))
ax.set_title('Top 12 Most Popular Genres on Netflix', fontsize=16, fontweight='bold')
ax.set_xlabel('Genre', fontsize=12)
ax.set_ylabel('Number of Titles', fontsize=12)
plt.xticks(rotation=45, ha='right', fontsize=10)

# Add count labels on top of each bar
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.1,
            f'{int(height)}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('chart_genres.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved as chart_genres.png')

---
## Step 6: Ratings Analysis

Let's look at how Netflix content is rated by viewers.
We have two ratings:
- **IMDb Rating**: Rating from IMDb website (scale: 0-10)
- **Netflix Rating**: Rating on Netflix platform (scale: 0-5)

In [ ]:
# Basic statistics for ratings
print('📌 IMDb Rating Statistics:')
print(df['imdb_rating'].describe().round(2))

print('\n📌 Netflix Rating Statistics:')
print(df['netflix_rating'].describe().round(2))

In [ ]:
# Compare ratings between TV Shows and Movies
print('📌 Average IMDb Rating by Content Type:')
avg_ratings = df.groupby('type')['imdb_rating'].mean().round(2)
print(avg_ratings)

# Create a Box Plot to compare ratings
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Box plot for IMDb ratings
df.boxplot(column='imdb_rating', by='type', ax=axes[0],
           boxprops=dict(color='#E50914'),
           whiskerprops=dict(color='#E50914'),
           medianprops=dict(color='darkred', linewidth=2),
           capprops=dict(color='#E50914'))
axes[0].set_title('IMDb Rating Distribution\nby Content Type', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Content Type', fontsize=11)
axes[0].set_ylabel('IMDb Rating (0-10)', fontsize=11)
plt.sca(axes[0])

# Box plot for Netflix ratings
df.boxplot(column='netflix_rating', by='type', ax=axes[1],
           boxprops=dict(color='#E50914'),
           whiskerprops=dict(color='#E50914'),
           medianprops=dict(color='darkred', linewidth=2),
           capprops=dict(color='#E50914'))
axes[1].set_title('Netflix Rating Distribution\nby Content Type', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Content Type', fontsize=11)
axes[1].set_ylabel('Netflix Rating (0-5)', fontsize=11)
plt.sca(axes[1])

plt.suptitle('')  # Remove auto-generated suptitle
plt.tight_layout()
plt.savefig('chart_ratings_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved as chart_ratings_boxplot.png')

In [ ]:
# Distribution of IMDb Ratings (Histogram)
fig, ax = plt.subplots(figsize=(10, 6))

ax.hist(df['imdb_rating'], bins=15, color='#E50914', edgecolor='white', alpha=0.8)
ax.axvline(df['imdb_rating'].mean(), color='black', linestyle='--',
           linewidth=2, label=f'Average: {df["imdb_rating"].mean():.1f}')
ax.set_title('Distribution of IMDb Ratings', fontsize=16, fontweight='bold')
ax.set_xlabel('IMDb Rating', fontsize=12)
ax.set_ylabel('Number of Titles', fontsize=12)
ax.legend(fontsize=12)

plt.tight_layout()
plt.savefig('chart_imdb_histogram.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved as chart_imdb_histogram.png')

In [ ]:
# Top 10 Highest Rated Titles
print('📌 Top 10 Highest Rated Titles (by IMDb):')
top_rated = df.nlargest(10, 'imdb_rating')[['title', 'type', 'genre', 'imdb_rating', 'views_millions']]
print(top_rated.to_string(index=False))

# Create a Horizontal Bar Chart
fig, ax = plt.subplots(figsize=(12, 7))
colors_list = plt.cm.RdYlGn(np.linspace(0.4, 0.9, 10))
bars = ax.barh(top_rated['title'], top_rated['imdb_rating'],
               color=colors_list)
ax.set_title('Top 10 Highest Rated Netflix Titles (IMDb)', fontsize=15, fontweight='bold')
ax.set_xlabel('IMDb Rating', fontsize=12)
ax.set_xlim(0, 10)

# Add rating labels
for bar, val in zip(bars, top_rated['imdb_rating']):
    ax.text(val + 0.05, bar.get_y() + bar.get_height()/2,
            f'{val}', va='center', fontsize=11, fontweight='bold')

ax.invert_yaxis()  # Highest rating at top
plt.tight_layout()
plt.savefig('chart_top_rated.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved as chart_top_rated.png')

---
## Step 7: Views Analysis

Let's analyze how many people watched different titles.
Views are measured in **millions of hours** watched.

In [ ]:
# Basic statistics for views
print('📌 Views Statistics (in Millions):')
print(df['views_millions'].describe().round(1))
print(f'\n🏆 Most viewed title: {df.loc[df["views_millions"].idxmax(), "title"]} '
      f'({df["views_millions"].max():.1f}M views)')
print(f'📉 Least viewed title: {df.loc[df["views_millions"].idxmin(), "title"]} '
      f'({df["views_millions"].min():.1f}M views)')

In [ ]:
# Top 10 Most Viewed Titles
top_viewed = df.nlargest(10, 'views_millions')[['title', 'type', 'genre', 'views_millions', 'imdb_rating']]
print('📌 Top 10 Most Viewed Netflix Titles:')
print(top_viewed.to_string(index=False))

# Create a Horizontal Bar Chart
fig, ax = plt.subplots(figsize=(12, 7))
colors_list = plt.cm.Blues(np.linspace(0.4, 0.9, 10))
bars = ax.barh(top_viewed['title'], top_viewed['views_millions'], color=colors_list)
ax.set_title('Top 10 Most Viewed Netflix Titles', fontsize=15, fontweight='bold')
ax.set_xlabel('Views (Millions)', fontsize=12)

# Add view count labels
for bar, val in zip(bars, top_viewed['views_millions']):
    ax.text(val + 5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}M', va='center', fontsize=10, fontweight='bold')

ax.invert_yaxis()
plt.tight_layout()
plt.savefig('chart_top_viewed.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved as chart_top_viewed.png')

In [ ]:
# Compare average views between TV Shows and Movies
print('📌 Average Views by Content Type:')
avg_views = df.groupby('type')['views_millions'].agg(['mean', 'sum', 'max']).round(1)
avg_views.columns = ['Average Views (M)', 'Total Views (M)', 'Max Views (M)']
print(avg_views)

# Bar chart comparing views
fig, ax = plt.subplots(figsize=(8, 6))
content_types = avg_views.index
bars = ax.bar(content_types, avg_views['Average Views (M)'],
              color=['#E50914', '#221F1F'], edgecolor='white', linewidth=1.5)
ax.set_title('Average Views by Content Type', fontsize=15, fontweight='bold')
ax.set_xlabel('Content Type', fontsize=12)
ax.set_ylabel('Average Views (Millions)', fontsize=12)

for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 1,
            f'{height:.1f}M', ha='center', va='bottom', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('chart_views_by_type.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved as chart_views_by_type.png')

---
## Step 8: TV Shows - Season Analysis

Let's specifically look at TV shows and their seasons.
This helps us understand which shows keep viewers coming back for more!

In [ ]:
# Focus on TV Shows only
print('📌 TV Shows by Season Number:')
season_counts = tv_shows['season'].value_counts().sort_index()
print(season_counts)

# Bar chart of season numbers
fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(season_counts.index.astype(str), season_counts.values,
       color=sns.color_palette('Set2', len(season_counts)))
ax.set_title('Number of TV Shows by Season Number', fontsize=15, fontweight='bold')
ax.set_xlabel('Season Number', fontsize=12)
ax.set_ylabel('Number of Shows', fontsize=12)

for i, (x, y) in enumerate(zip(season_counts.index, season_counts.values)):
    ax.text(i, y + 0.1, str(y), ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('chart_season_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved as chart_season_distribution.png')

In [ ]:
# Does being in a later season affect ratings?
print('📌 Average IMDb Rating by Season:')
season_ratings = tv_shows.groupby('season')['imdb_rating'].mean().round(2)
print(season_ratings)

# Line chart showing rating trend over seasons
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(season_ratings.index, season_ratings.values,
        marker='o', linewidth=2.5, markersize=10, color='#E50914')
ax.fill_between(season_ratings.index, season_ratings.values,
                alpha=0.1, color='#E50914')

# Add rating labels on each point
for x, y in zip(season_ratings.index, season_ratings.values):
    ax.annotate(f'{y:.1f}', (x, y), textcoords='offset points',
                xytext=(0, 12), ha='center', fontsize=11, fontweight='bold')

ax.set_title('Average IMDb Rating vs Season Number', fontsize=15, fontweight='bold')
ax.set_xlabel('Season Number', fontsize=12)
ax.set_ylabel('Average IMDb Rating', fontsize=12)
ax.set_ylim(5, 10)
ax.set_xticks(season_ratings.index)
plt.tight_layout()
plt.savefig('chart_rating_by_season.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved as chart_rating_by_season.png')

In [ ]:
# Does season number affect views?
print('📌 Average Views by Season Number:')
season_views = tv_shows.groupby('season')['views_millions'].mean().round(1)
print(season_views)

# Bar chart
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(season_views.index.astype(str), season_views.values,
              color=sns.color_palette('Blues_r', len(season_views)))
ax.set_title('Average Views by Season Number (TV Shows)', fontsize=15, fontweight='bold')
ax.set_xlabel('Season Number', fontsize=12)
ax.set_ylabel('Average Views (Millions)', fontsize=12)

for bar, val in zip(bars, season_views.values):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 5,
            f'{val:.0f}M', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('chart_views_by_season.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved as chart_views_by_season.png')

---
## Step 9: Genre Analysis

Which genres get the best ratings and most views?

In [ ]:
# Extract primary genre (first genre before '/')
df['primary_genre'] = df['genre'].apply(lambda x: x.split('/')[0].strip())

# Average rating by genre
genre_stats = df.groupby('primary_genre').agg(
    count=('title', 'count'),
    avg_imdb=('imdb_rating', 'mean'),
    avg_views=('views_millions', 'mean')
).round(2)

# Only keep genres with 2+ titles
genre_stats = genre_stats[genre_stats['count'] >= 2].sort_values('avg_imdb', ascending=False)

print('📌 Genre Statistics (genres with 2+ titles):')
print(genre_stats.to_string())

In [ ]:
# Scatter plot: Rating vs Views by Genre
fig, ax = plt.subplots(figsize=(12, 8))

# Get unique primary genres and assign colors
unique_genres = df['primary_genre'].unique()
color_map = plt.cm.tab20(np.linspace(0, 1, len(unique_genres)))
genre_colors = dict(zip(unique_genres, color_map))

# Plot each title
for genre in unique_genres:
    mask = df['primary_genre'] == genre
    ax.scatter(df.loc[mask, 'imdb_rating'],
               df.loc[mask, 'views_millions'],
               c=[genre_colors[genre]], label=genre, s=80, alpha=0.8)

# Label the top 5 viewed titles
top5 = df.nlargest(5, 'views_millions')
for _, row in top5.iterrows():
    ax.annotate(row['title'], (row['imdb_rating'], row['views_millions']),
                textcoords='offset points', xytext=(5, 5), fontsize=8)

ax.set_title('IMDb Rating vs Views (by Primary Genre)', fontsize=15, fontweight='bold')
ax.set_xlabel('IMDb Rating', fontsize=12)
ax.set_ylabel('Views (Millions)', fontsize=12)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig('chart_rating_vs_views.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved as chart_rating_vs_views.png')

---
## Step 10: Country & Language Analysis

Netflix is a global platform. Let's see which countries produce the most content.

In [ ]:
# Extract primary country
df['primary_country'] = df['country'].apply(lambda x: x.split('/')[0].strip())

# Count by country
country_counts = df['primary_country'].value_counts().head(10)
print('📌 Top Countries by Number of Titles:')
print(country_counts)

# Bar chart
fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(country_counts.index, country_counts.values,
              color=sns.color_palette('Set3', len(country_counts)))
ax.set_title('Top Countries by Number of Netflix Titles', fontsize=15, fontweight='bold')
ax.set_xlabel('Country', fontsize=12)
ax.set_ylabel('Number of Titles', fontsize=12)

for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.1,
            str(int(height)), ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('chart_country.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved as chart_country.png')

In [ ]:
# Does non-English content get as many views?
df['is_english'] = df['language'].apply(lambda x: 'English' if 'English' in x else 'Non-English')

lang_stats = df.groupby('is_english').agg(
    count=('title', 'count'),
    avg_views=('views_millions', 'mean'),
    avg_rating=('imdb_rating', 'mean')
).round(2)

print('📌 English vs Non-English Content Performance:')
print(lang_stats)

# Side by side comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Views comparison
bars1 = axes[0].bar(lang_stats.index, lang_stats['avg_views'],
                    color=['#E50914', '#B81D24'])
axes[0].set_title('Average Views: English vs Non-English', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Average Views (Millions)', fontsize=11)
for bar in bars1:
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                f'{bar.get_height():.1f}M', ha='center', va='bottom', fontsize=12, fontweight='bold')

# Rating comparison
bars2 = axes[1].bar(lang_stats.index, lang_stats['avg_rating'],
                    color=['#221F1F', '#564D4D'])
axes[1].set_title('Average IMDb Rating: English vs Non-English', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Average IMDb Rating', fontsize=11)
axes[1].set_ylim(0, 10)
for bar in bars2:
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.1,
                f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('chart_language_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved as chart_language_comparison.png')

---
## Step 11: Correlation Analysis

**Correlation** tells us if two things are related.
- A value close to **+1** means they increase together (positive correlation)
- A value close to **-1** means when one increases, the other decreases
- A value close to **0** means no relationship

In [ ]:
# Select numerical columns for correlation
numerical_cols = ['imdb_rating', 'netflix_rating', 'views_millions',
                  'duration_minutes', 'season', 'release_year']
corr_data = df[numerical_cols].copy()

# Calculate correlation matrix
correlation_matrix = corr_data.corr().round(2)
print('📌 Correlation Matrix:')
print(correlation_matrix)

# Create a Heatmap
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    correlation_matrix,
    annot=True,          # Show numbers in cells
    fmt='.2f',           # Format numbers to 2 decimal places
    cmap='RdYlGn',       # Red-Yellow-Green color scheme
    center=0,            # Center the color at 0
    vmin=-1, vmax=1,
    ax=ax,
    square=True,
    linewidths=0.5
)
ax.set_title('Correlation Heatmap of Netflix Metrics', fontsize=15, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('chart_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved as chart_correlation_heatmap.png')

# Interpretation
print('\n📌 Key Insight:')
print(f'  Correlation between IMDb Rating and Views: {correlation_matrix.loc["imdb_rating", "views_millions"]:.2f}')
print(f'  Correlation between IMDb and Netflix Rating: {correlation_matrix.loc["imdb_rating", "netflix_rating"]:.2f}')

---
## Step 12: Final Dashboard - All Key Charts Together

Let's create one final chart that shows all the important findings together!

In [ ]:
fig = plt.figure(figsize=(20, 18))
fig.suptitle('Netflix Content Analysis Dashboard', fontsize=22, fontweight='bold', y=1.01)

# --- Chart 1: Content type pie chart (top-left) ---
ax1 = fig.add_subplot(3, 3, 1)
content_counts = df['type'].value_counts()
ax1.pie(content_counts.values, labels=content_counts.index,
        autopct='%1.1f%%', colors=['#E50914', '#221F1F'],
        wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
ax1.set_title('Content Types', fontsize=12, fontweight='bold')

# --- Chart 2: Top 5 most viewed (top-center) ---
ax2 = fig.add_subplot(3, 3, 2)
top5v = df.nlargest(5, 'views_millions')
ax2.barh(top5v['title'].str[:25], top5v['views_millions'], color=plt.cm.Reds(np.linspace(0.5, 0.9, 5)))
ax2.set_title('Top 5 Most Viewed', fontsize=12, fontweight='bold')
ax2.set_xlabel('Views (M)')
ax2.invert_yaxis()

# --- Chart 3: Top 5 highest rated (top-right) ---
ax3 = fig.add_subplot(3, 3, 3)
top5r = df.nlargest(5, 'imdb_rating')
ax3.barh(top5r['title'].str[:25], top5r['imdb_rating'], color=plt.cm.Greens(np.linspace(0.5, 0.9, 5)))
ax3.set_title('Top 5 Highest Rated (IMDb)', fontsize=12, fontweight='bold')
ax3.set_xlabel('IMDb Rating')
ax3.set_xlim(0, 10)
ax3.invert_yaxis()

# --- Chart 4: Rating distribution histogram (middle-left) ---
ax4 = fig.add_subplot(3, 3, 4)
ax4.hist(df['imdb_rating'], bins=12, color='#E50914', edgecolor='white', alpha=0.8)
ax4.axvline(df['imdb_rating'].mean(), color='black', linestyle='--', linewidth=2)
ax4.set_title('IMDb Rating Distribution', fontsize=12, fontweight='bold')
ax4.set_xlabel('IMDb Rating')
ax4.set_ylabel('Count')

# --- Chart 5: Views by season (middle-center) ---
ax5 = fig.add_subplot(3, 3, 5)
season_views_plot = tv_shows.groupby('season')['views_millions'].mean()
ax5.bar(season_views_plot.index.astype(str), season_views_plot.values,
        color=sns.color_palette('Blues_r', len(season_views_plot)))
ax5.set_title('Avg Views by Season (TV Shows)', fontsize=12, fontweight='bold')
ax5.set_xlabel('Season #')
ax5.set_ylabel('Avg Views (M)')

# --- Chart 6: Rating by season (middle-right) ---
ax6 = fig.add_subplot(3, 3, 6)
season_ratings_plot = tv_shows.groupby('season')['imdb_rating'].mean()
ax6.plot(season_ratings_plot.index, season_ratings_plot.values,
         marker='o', color='#E50914', linewidth=2, markersize=8)
ax6.set_title('Avg IMDb Rating by Season', fontsize=12, fontweight='bold')
ax6.set_xlabel('Season #')
ax6.set_ylabel('Avg IMDb Rating')
ax6.set_xticks(season_ratings_plot.index)

# --- Chart 7: Scatter - rating vs views (bottom-left) ---
ax7 = fig.add_subplot(3, 3, 7)
for content_type, color in [('TV Show', '#E50914'), ('Movie', '#221F1F')]:
    mask = df['type'] == content_type
    ax7.scatter(df.loc[mask, 'imdb_rating'], df.loc[mask, 'views_millions'],
                c=color, label=content_type, alpha=0.7, s=50)
ax7.set_title('Rating vs Views', fontsize=12, fontweight='bold')
ax7.set_xlabel('IMDb Rating')
ax7.set_ylabel('Views (M)')
ax7.legend(fontsize=9)

# --- Chart 8: Top genres (bottom-center) ---
ax8 = fig.add_subplot(3, 3, 8)
top_genres = genre_df.head(8)
ax8.bar(top_genres['Genre'], top_genres['Count'],
        color=sns.color_palette('Set2', len(top_genres)))
ax8.set_title('Top 8 Genres', fontsize=12, fontweight='bold')
ax8.set_xlabel('Genre')
ax8.set_ylabel('Count')
plt.setp(ax8.xaxis.get_majorticklabels(), rotation=45, ha='right', fontsize=8)

# --- Chart 9: Avg views by content type (bottom-right) ---
ax9 = fig.add_subplot(3, 3, 9)
type_views = df.groupby('type')['views_millions'].mean()
bars9 = ax9.bar(type_views.index, type_views.values,
                color=['#E50914', '#221F1F'], edgecolor='white', linewidth=1.5)
ax9.set_title('Avg Views by Content Type', fontsize=12, fontweight='bold')
ax9.set_ylabel('Avg Views (M)')
for bar in bars9:
    ax9.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
             f'{bar.get_height():.1f}M', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('chart_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Dashboard saved as chart_dashboard.png')

---
## Step 13: Key Findings & Conclusions

Let's summarize everything we found!

In [ ]:
# ============================================================
#  SUMMARY OF KEY FINDINGS
# ============================================================

print('=' * 60)
print('       NETFLIX DATA ANALYSIS - KEY FINDINGS')
print('=' * 60)

print(f"""
📊 DATASET OVERVIEW
   Total Titles Analyzed : {len(df)}
   TV Shows              : {len(tv_shows)}
   Movies                : {len(movies)}

⭐ RATINGS
   Average IMDb Rating   : {df['imdb_rating'].mean():.2f} / 10
   Highest Rated Title   : {df.loc[df['imdb_rating'].idxmax(), 'title']} ({df['imdb_rating'].max()})
   Lowest Rated Title    : {df.loc[df['imdb_rating'].idxmin(), 'title']} ({df['imdb_rating'].min()})

👁️ VIEWS
   Total Views           : {df['views_millions'].sum():.1f} Million
   Most Viewed Title     : {df.loc[df['views_millions'].idxmax(), 'title']} ({df['views_millions'].max():.1f}M)
   Avg Views (TV Shows)  : {tv_shows['views_millions'].mean():.1f}M
   Avg Views (Movies)    : {movies['views_millions'].mean():.1f}M

📺 TV SHOWS - SEASON INSIGHTS
   Most Seasons Available: {tv_shows['season'].max()} seasons
   Season 1 Avg Views    : {tv_shows[tv_shows['season']==1]['views_millions'].mean():.1f}M
   Best Rated Season     : Season {season_ratings.idxmax()} (avg {season_ratings.max():.2f})

🌍 COUNTRIES
   Most Content From     : {df['primary_country'].value_counts().index[0]}
"""
)
print('=' * 60)
print()
print('🎯 CONCLUSIONS:')
print('   1. TV Shows attract significantly more views than Movies')
print('   2. Higher IMDb ratings generally correlate with more views')
print('   3. Season 1 shows tend to get the most views (new shows draw curiosity)')
print('   4. Drama and Action are the most common genres on Netflix')
print('   5. USA produces the most Netflix content, but global shows are gaining popularity')
print('   6. Non-English shows (like Squid Game) can rival English shows in viewership')
print('=' * 60)

---
## 🎉 Congratulations!

You have completed your first data science project! Here's a recap of what you learned:

| Skill | What You Did |
|-------|-------------|
| **Data Loading** | Loaded a CSV file using `pd.read_csv()` |
| **Data Exploration** | Used `.head()`, `.info()`, `.describe()` |
| **Data Cleaning** | Checked for missing values and duplicates |
| **Data Analysis** | Used `.groupby()`, `.value_counts()`, `.mean()` |
| **Visualization** | Created bar charts, pie charts, histograms, scatter plots, heatmaps |
| **Insights** | Drew conclusions from the data |

### 🚀 Next Steps to Level Up:
1. Try modifying the dataset with your own data
2. Learn about **machine learning** to predict ratings
3. Try **web scraping** to collect real Netflix/IMDb data
4. Learn **SQL** to work with databases
5. Explore **Tableau** or **Power BI** for interactive dashboards

---
*Happy coding! 🐍📊*